# Automated Web Security Scanner Using Python

**Student:** Sindhu T Byali  
**Project type:** Internship project  
**Purpose:** A lightweight scanner that reviews basic HTTP security configuration for websites you own or have explicit permission to assess.

## Features
- Validates target URLs and blocks obvious localhost/private-network targets.
- Checks HTTPS use and common HTTP security headers.
- Reviews basic CORS response behavior using a harmless test Origin.
- Identifies cookie security attributes when cookies are returned.
- Produces a findings table and saves a CSV report.

## Responsible-use note
Only scan websites you own or are explicitly authorized to test. This is a basic configuration checker, not a penetration-testing tool. It does not exploit vulnerabilities and cannot prove that a website is secure. Network/DNS environments can change; use it only in a controlled, authorized context.

## 1. Install dependencies

Run this cell once in your notebook environment. If your environment requires a kernel restart after installing packages, restart the kernel and continue.

In [ ]:
%pip install -q requests pandas

## 2. Import libraries and define safety checks

The scanner accepts only HTTP/HTTPS URLs and rejects obvious loopback, private, link-local, and reserved IP destinations. It does not automatically follow redirects, so each redirect target can be reviewed before scanning it separately.

In [ ]:
import ipaddress
import socket
from urllib.parse import urlparse, urljoin
from datetime import datetime, timezone

import pandas as pd
import requests
from IPython.display import display

SECURITY_HEADERS = {
    "Strict-Transport-Security": "Helps browsers enforce HTTPS after a secure visit.",
    "Content-Security-Policy": "Helps restrict the sources from which a page can load content.",
    "X-Content-Type-Options": "Helps prevent MIME-type sniffing; commonly set to nosniff.",
    "X-Frame-Options": "Can reduce clickjacking risk in browsers that honor it.",
    "Referrer-Policy": "Controls how much referrer information browsers send.",
    "Permissions-Policy": "Controls access to selected browser features.",
}

REQUEST_TIMEOUT_SECONDS = 10
MAX_RESPONSE_BYTES = 1_000_000  # Read at most 1 MB of response body.
USER_AGENT = "Internship-WebSecurityScanner/1.0 (authorized configuration audit)"


def _host_resolves_only_to_public_ips(hostname):
    """Best-effort DNS/IP safety check. This is not a substitute for authorization."""
    try:
        literal_ip = ipaddress.ip_address(hostname.strip("[]"))
        return literal_ip.is_global
    except ValueError:
        pass

    try:
        records = socket.getaddrinfo(hostname, None, type=socket.SOCK_STREAM)
        addresses = {record[4][0] for record in records}
        if not addresses:
            return False
        return all(ipaddress.ip_address(addr.split("%")[0]).is_global for addr in addresses)
    except (socket.gaierror, ValueError, OSError):
        return False


def validate_target_url(url):
    """Return a normalized URL or raise ValueError if it fails basic safety checks."""
    if not isinstance(url, str) or not url.strip():
        raise ValueError("Enter a website URL, for example https://example.com")

    candidate = url.strip()
    if "://" not in candidate:
        candidate = "https://" + candidate

    parsed = urlparse(candidate)
    if parsed.scheme.lower() not in {"http", "https"}:
        raise ValueError("Only http:// and https:// URLs are supported.")
    if not parsed.hostname:
        raise ValueError("The URL must include a valid hostname.")
    if parsed.username or parsed.password:
        raise ValueError("URLs containing embedded usernames/passwords are not accepted.")
    if parsed.port and not (1 <= parsed.port <= 65535):
        raise ValueError("The URL port is invalid.")
    if parsed.hostname.lower() in {"localhost", "localhost.localdomain"}:
        raise ValueError("Localhost targets are blocked.")
    if not _host_resolves_only_to_public_ips(parsed.hostname):
        raise ValueError("The host did not resolve exclusively to public IP addresses. Private, local, reserved, or unresolved targets are blocked.")

    # Keep the URL without fragments; fragments are not sent to the server.
    return parsed._replace(fragment="").geturl()


def _classify_header(header_name, header_value, is_https):
    if header_name == "Strict-Transport-Security" and not is_https:
        return "INFO", "HSTS is meaningful on HTTPS responses; this target used HTTP."
    if header_name == "X-Content-Type-Options":
        if header_value.strip().lower() == "nosniff":
            return "PASS", "Header is set to nosniff."
        return "WARNING", "Header is present but is not set to nosniff."
    if header_name == "Content-Security-Policy":
        return "PASS", "Content-Security-Policy header is present; policy quality was not deeply evaluated."
    if header_name in {"X-Frame-Options", "Referrer-Policy", "Permissions-Policy", "Strict-Transport-Security"}:
        return "PASS", "Header is present; its full policy quality was not deeply evaluated."
    return "INFO", "Header observed."


def scan_website(url):
    """
    Perform a limited, non-invasive HTTP configuration check.
    Redirects are not followed automatically. Only scan authorized public targets.
    """
    target = validate_target_url(url)
    parsed = urlparse(target)
    is_https = parsed.scheme.lower() == "https"

    findings = []
    result = {
        "target_url": target,
        "scan_time_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "http_status": None,
        "final_url": target,
        "redirect_location": None,
        "error": None,
    }

    headers = {
        "User-Agent": USER_AGENT,
        "Origin": "https://scanner-audit.example",
        "Accept": "text/html,application/xhtml+xml,application/json;q=0.9,*/*;q=0.8",
    }

    try:
        # Do not follow redirects automatically; this avoids silently scanning a different host.
        with requests.get(
            target,
            headers=headers,
            timeout=REQUEST_TIMEOUT_SECONDS,
            allow_redirects=False,
            stream=True,
        ) as response:
            result["http_status"] = response.status_code
            response_headers = response.headers
            location = response_headers.get("Location")
            if response.is_redirect and location:
                result["redirect_location"] = urljoin(target, location)
                findings.append({
                    "check": "Redirect",
                    "status": "INFO",
                    "finding": f"Server returned a redirect to {result['redirect_location']}. Review the destination and scan it separately if authorized.",
                    "evidence": f"HTTP {response.status_code}",
                })

            # HTTPS use
            if is_https:
                findings.append({
                    "check": "HTTPS",
                    "status": "PASS",
                    "finding": "Target URL uses HTTPS. This check does not independently audit TLS configuration.",
                    "evidence": "URL scheme is https",
                })
            else:
                findings.append({
                    "check": "HTTPS",
                    "status": "WARNING",
                    "finding": "Target URL uses HTTP rather than HTTPS.",
                    "evidence": "URL scheme is http",
                })

            # HTTP status
            if 200 <= response.status_code < 400:
                status = "PASS" if response.status_code < 300 else "INFO"
                message = f"Server responded with HTTP {response.status_code}."
            elif response.status_code in {401, 403, 404, 405}:
                status = "INFO"
                message = f"Server responded with HTTP {response.status_code}; access restrictions or endpoint behavior may explain this."
            else:
                status = "WARNING"
                message = f"Server responded with HTTP {response.status_code}; investigate if unexpected."
            findings.append({
                "check": "HTTP response",
                "status": status,
                "finding": message,
                "evidence": f"HTTP {response.status_code}",
            })

            # Security headers
            for header_name, description in SECURITY_HEADERS.items():
                value = response_headers.get(header_name)
                if value:
                    status, message = _classify_header(header_name, value, is_https)
                    findings.append({
                        "check": f"Header: {header_name}",
                        "status": status,
                        "finding": message,
                        "evidence": f"{header_name}: {value[:250]}",
                    })
                else:
                    severity = "WARNING"
                    if header_name == "Strict-Transport-Security" and not is_https:
                        severity = "INFO"
                    findings.append({
                        "check": f"Header: {header_name}",
                        "status": severity,
                        "finding": f"{header_name} was not observed. {description}",
                        "evidence": "Header absent from response",
                    })

            # CORS: only a basic response behavior check, not a full CORS audit.
            allow_origin = response_headers.get("Access-Control-Allow-Origin")
            allow_credentials = response_headers.get("Access-Control-Allow-Credentials", "").strip().lower()
            test_origin = headers["Origin"]
            if allow_origin == "*":
                findings.append({
                    "check": "CORS",
                    "status": "INFO",
                    "finding": "Access-Control-Allow-Origin is '*'. This is not automatically a vulnerability; risk depends on the resource and credential behavior.",
                    "evidence": f"Access-Control-Allow-Origin: {allow_origin}; credentials: {allow_credentials or 'not set'}",
                })
            elif allow_origin == test_origin and allow_credentials == "true":
                findings.append({
                    "check": "CORS",
                    "status": "WARNING",
                    "finding": "The response reflected the test Origin and allowed credentials. Review the CORS allowlist and intended access model.",
                    "evidence": f"Origin reflected; Access-Control-Allow-Credentials: {allow_credentials}",
                })
            elif allow_origin:
                findings.append({
                    "check": "CORS",
                    "status": "INFO",
                    "finding": "An Access-Control-Allow-Origin header is present. This single response does not establish whether the CORS policy is safe.",
                    "evidence": f"Access-Control-Allow-Origin: {allow_origin[:250]}",
                })
            else:
                findings.append({
                    "check": "CORS",
                    "status": "PASS",
                    "finding": "No Access-Control-Allow-Origin header was observed in this response. CORS behavior may differ by route or request.",
                    "evidence": "Header absent from response",
                })

            # Cookie flags
            set_cookie = response_headers.get("Set-Cookie")
            if set_cookie:
                cookie_lower = set_cookie.lower()
                for flag, present, recommendation in [
                    ("Secure", "secure" in cookie_lower, "Secure"),
                    ("HttpOnly", "httponly" in cookie_lower, "HttpOnly"),
                    ("SameSite", "samesite=" in cookie_lower, "SameSite"),
                ]:
                    findings.append({
                        "check": f"Cookie attribute: {flag}",
                        "status": "PASS" if present else "WARNING",
                        "finding": f"{recommendation} attribute was {'observed' if present else 'not observed'} in the sampled Set-Cookie header. Multiple cookies may have different settings.",
                        "evidence": f"Set-Cookie: {set_cookie[:250]}",
                    })
            else:
                findings.append({
                    "check": "Cookies",
                    "status": "INFO",
                    "finding": "No Set-Cookie header was observed in this response.",
                    "evidence": "Set-Cookie absent",
                })

            # Close response without downloading a potentially large body.
            result["final_url"] = target

    except requests.exceptions.SSLError as exc:
        result["error"] = f"TLS/SSL error: {exc.__class__.__name__}"
        findings.append({
            "check": "Connection",
            "status": "FAIL",
            "finding": "The HTTPS connection failed certificate/TLS validation. Do not disable certificate verification to hide this result.",
            "evidence": result["error"],
        })
    except requests.exceptions.Timeout:
        result["error"] = "Request timed out"
        findings.append({
            "check": "Connection",
            "status": "WARNING",
            "finding": "The request timed out.",
            "evidence": result["error"],
        })
    except requests.exceptions.RequestException as exc:
        result["error"] = f"Request error: {exc.__class__.__name__}"
        findings.append({
            "check": "Connection",
            "status": "FAIL",
            "finding": "The request could not be completed.",
            "evidence": result["error"],
        })
    except ValueError as exc:
        raise

    findings_df = pd.DataFrame(findings, columns=["check", "status", "finding", "evidence"])
    return result, findings_df

## 3. Run a scan

Replace the example URL with a website you own or have explicit permission to assess. If the server redirects, review the reported destination and scan it separately only if authorized.

In [ ]:
# Example:
# target_url = "https://your-authorized-test-site.example"
# scan_summary, findings = scan_website(target_url)
# print(scan_summary)
# display(findings)

target_url = input("Enter an authorized public website URL (https://...): ").strip()
try:
    scan_summary, findings = scan_website(target_url)
    print("Scan summary:")
    display(pd.DataFrame([scan_summary]))
    print("Findings:")
    display(findings)
except ValueError as exc:
    print(f"Input validation: {exc}")

## 4. Export findings

After running the scan cell successfully, run this cell to save the findings as a CSV file in the notebook's current working directory.

In [ ]:
if "findings" in globals() and isinstance(findings, pd.DataFrame):
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    report_file = f"web_security_findings_{timestamp}.csv"
    findings.to_csv(report_file, index=False)
    print(f"Saved findings to: {report_file}")
else:
    print("Run the scan cell first to create findings.")

## 5. Interpretation and limitations

- **PASS** means the specific limited check passed; it does not mean the whole website is secure.
- **WARNING** indicates a configuration item worth reviewing, not proof of exploitability.
- **INFO** is contextual and may be acceptable depending on the application.
- Header presence is not equivalent to a high-quality policy. For example, a Content Security Policy needs a separate review of its directives.
- CORS behavior can differ by path, method, Origin, authentication state, and preflight request. This tool performs only a basic response check.
- The tool does not crawl pages, exploit vulnerabilities, test authentication bypasses, or perform load testing.
- Use the scanner only with permission and follow your organization's rules of engagement.